# PGD Adversarial Training on CIFAR-10 (Kaggle T4)Trains a ResNet-18 with projected gradient descent adversarial training (Madry et al., ICLR 2018), then evaluates theselected checkpoint under the same protocol as the naturally trained baseline and runs the obfuscated-gradient sanitychecks on the result.**Prerequisites.**1. **Accelerator: GPU.** Cell 1 fails loudly on CPU. A k-step PGD attack needs k forward and backward passes per batch   plus one for the parameter update, so this costs roughly eight times standard training: one to two hours on a T4,   close to two days on CPU.2. **Internet: on**, so CIFAR-10 can be downloaded once.3. **Toolkit dataset attached.** Zip the `adversarial-ml-toolkit` directory so that `attacks/`, `defenses/`,   `experiments/` and `models/` sit at the top level of the archive, and upload it as a private Kaggle Dataset. Cell 4   locates it automatically.**Why the code is imported rather than inlined.** A pasted copy of the attack and training code would be a secondversion with no mechanism keeping it equal to the reviewed one. This project has repeatedly hit exactly that failure,including a PGD update that used a multiply in place of an addition and still passed every contract test.**What is selected on.** Checkpoint selection uses robust accuracy on a 5,000-image split held out of the training set.The test set is untouched until the final evaluation cell. Selecting on the test set would leak test information intomodel selection and the reported number would no longer be held out.**Recovery.** There is no resume path. The history CSV is flushed every epoch and the best checkpoint is rewrittenwhenever it improves, both under `/kaggle/working`, so an interrupted session leaves usable artefacts, but continuingfrom epoch 25 is not possible.

## 1. Environment

In [ ]:
import osimport platformimport sysfrom pathlib import Pathimport torchif not torch.cuda.is_available():    raise RuntimeError(        "No CUDA device. Set Accelerator to GPU in the notebook settings. This run costs roughly eight times "        "standard training and is not viable on CPU."    )DEVICE = torch.device("cuda")print(f"python       {platform.python_version()}")print(f"torch        {torch.__version__}  (cuda {torch.version.cuda})")print(f"gpu          {torch.cuda.get_device_name(0)}, "      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")print(f"cpu count    {os.cpu_count()}")

## 2. Configuration`SMOKE` runs a two-epoch job on 1,000 training images to prove the path works before committing GPU hours. Run it once,confirm the verification cell lists every file, then set `SMOKE = False` and run again. The two modes write to separatedirectories so a smoke checkpoint of near-random weights can never be mistaken for a trained one.`TrainConfig` is the single source of truth for hyperparameters: it validates its own arguments and serialises itself toJSON beside the results, so there is no second copy to drift.**On the epoch budget.** Madry and Rice both train for 200 epochs with decays at 100 and 150. Thirty epochs with decaysat the same fractions is a demonstration budget: it is enough to show robust overfitting, since that appears shortlyafter the first decay, and it will land below published numbers. If GPU quota allows, 60 epochs gives a morerepresentative curve for roughly double the time.**On `num_workers`.** Two is conservative for a four-vCPU Kaggle instance. Raise it to 3 if the GPU appears to bewaiting on data; the Windows default of zero would leave the T4 idle.

In [ ]:
SMOKE = TrueWORKING_ROOT = Path("/kaggle/working")DATA_ROOT = Path("/kaggle/temp/data")  # scratch: not saved as notebook outputRUN_NAME = "smoke" if SMOKE else "pgd_at_30ep"OUT_DIR = WORKING_ROOT / "results" / RUN_NAMECKPT_DIR = WORKING_ROOT / "checkpoints" / RUN_NAME# Passed to TrainConfig; anything absent keeps the reviewed module default.REAL_OVERRIDES = {    "epochs": 30,    "batch_size": 128,    "num_workers": 2,    "device": "cuda",}SMOKE_OVERRIDES = {    "epochs": 2,    "batch_size": 128,    "attack_steps": 2,    "val_attack_steps": 2,    "val_size": 49000,        # leaves 1,000 training images    "val_eval_samples": 256,    "lr_milestones": (0.5,),    "num_workers": 2,    "device": "cuda",}OVERRIDES = SMOKE_OVERRIDES if SMOKE else REAL_OVERRIDESprint(f"mode      {'SMOKE' if SMOKE else 'FULL'}")print(f"outputs   {OUT_DIR}")print(f"ckpts     {CKPT_DIR}")

## 3. DataThe toolkit disables torchvision's downloader, because the certificate on the development machine is expired and thelocal copy is authoritative there. Rather than modify the reviewed module for this environment, the notebook downloadsinto `/kaggle/temp`, which produces the `cifar-10-batches-py` directory the module expects, then points`CIFAR10_DATA_ROOT` at it. Scratch rather than `/kaggle/working` because the extracted dataset is around 350 MB andwould otherwise be saved as notebook output on every commit.

In [ ]:
from torchvision.datasets import CIFAR10DATA_ROOT.mkdir(parents=True, exist_ok=True)CIFAR10(root=str(DATA_ROOT), train=True, download=True)CIFAR10(root=str(DATA_ROOT), train=False, download=True)batches_dir = DATA_ROOT / "cifar-10-batches-py"if not batches_dir.is_dir():    raise RuntimeError(f"Expected {batches_dir} after download; the toolkit resolves data through this directory.")os.environ["CIFAR10_DATA_ROOT"] = str(DATA_ROOT)print(f"CIFAR10_DATA_ROOT = {DATA_ROOT}")

## 4. ToolkitThe dataset directory is located by searching for `defenses/adversarial_training.py`, which tolerates both a flatarchive and one wrapped in an extra folder. The signature assertions and source hashes exist so that an outdated uploadfails in seconds rather than after an hour of training against a stale attack. Record the printed hashes with any resultyou report.`_common.py` anchors its default paths with `parents[2]`, which resolves to a path that does not exist here. That isharmless: the defaults are consulted only when the environment variables are absent, and `CIFAR10_DATA_ROOT` is set.

In [ ]:
import hashlibimport inspectINPUT_ROOT = Path("/kaggle/input")MARKER = Path("defenses/adversarial_training.py")candidates = [p for p in INPUT_ROOT.glob("*") if (p / MARKER).is_file()]candidates += [p for p in INPUT_ROOT.glob("*/*") if (p / MARKER).is_file()]if not candidates:    listing = sorted(p.name for p in INPUT_ROOT.iterdir()) if INPUT_ROOT.is_dir() else []    raise RuntimeError(        f"No attached dataset contains {MARKER}. Attach the toolkit dataset. Present under /kaggle/input: {listing}"    )TOOLKIT_ROOT = candidates[0]sys.path.insert(0, str(TOOLKIT_ROOT))from attacks.pgd import pgdfrom defenses.adversarial_training import TrainConfig, trainfrom experiments._common import build_loader, load_modelpgd_params = list(inspect.signature(pgd).parameters)assert pgd_params[:7] == ["model", "images", "labels", "eps", "alpha", "steps", "random_start"], pgd_paramsassert "lr_milestones" in inspect.signature(TrainConfig).parametersprint(f"toolkit   {TOOLKIT_ROOT}")for relative in ("attacks/pgd.py", "attacks/fgsm.py", "attacks/cw.py",                 "defenses/adversarial_training.py", "experiments/robustness_eval.py"):    digest = hashlib.sha256((TOOLKIT_ROOT / relative).read_bytes()).hexdigest()[:12]    print(f"  {digest}  {relative}")

## 5. Resolved configurationDecay epochs are printed rather than assumed. The milestones are fractions of the run length and Python rounds halves toeven, so at 30 epochs `round(0.75 * 30)` is 22 rather than 23.

In [ ]:
config = TrainConfig(out_dir=OUT_DIR, ckpt_dir=CKPT_DIR, **OVERRIDES)train_images = 50_000 - config.val_sizesteps_per_epoch = -(-train_images // config.batch_size)print(f"epochs         {config.epochs}")print(f"lr             {config.lr}, x{config.lr_gamma} at epochs {config.milestone_epochs()}")print(f"optimiser      SGD momentum={config.momentum} weight_decay={config.weight_decay}")print(f"threat model   eps={config.eps:.6f} ({round(config.eps * 255)}/255), alpha={config.alpha:.6f}")print(f"attack         {config.attack_steps} steps training, {config.val_attack_steps} steps validation")print(f"data           {train_images} train / {config.val_size} held out, {steps_per_epoch} steps per epoch")print(f"cost           ~{config.attack_steps + 1}x standard training")print(f"selection      best held-out robust accuracy over {config.val_eval_samples} images; test set untouched")

## 6. TrainAdversarial examples are generated in eval mode, so the attack's forward passes leave BatchNorm's running statisticsuntouched and the training-time threat model matches the evaluation harness. The consequence, expected rather thanfaulty, is that those statistics are estimated from the adversarial distribution alone and clean accuracy lands wellbelow the 93.43% a naturally trained model reaches.**Read `val_robust_acc` as a selection metric, not a robustness estimate.** It uses 10 PGD steps with no random start,which is deliberately weaker and cheaper than the reported evaluation, and consistent across epochs so that checkpointcomparison is free of attack noise. The number to quote comes from the evaluation cell below.**Expected shape.** Both accuracies climb slowly, improve sharply just after the first decay, then `val_robust_acc`peaks within a few epochs and drifts down while `train_adv_acc` keeps rising. That drift is robust overfitting (Rice,Wong and Kolter, ICML 2020) and is why the loop keeps the best checkpoint rather than the last.**Three failure signatures.** Robust accuracy collapsing toward zero while the loss keeps falling is catastrophicoverfitting, which should not occur with a multi-step adversary and would point at a bug. Both curves flat from epoch onepoints at the learning rate. And `val_robust_acc` sitting at `val_clean_acc` means the attack is not working rather thanthat the model is robust.

In [ ]:
import timestarted = time.perf_counter()result = train(config)elapsed = time.perf_counter() - startedprint(f"\nwall clock       {elapsed / 60:.1f} min ({elapsed / config.epochs:.0f}s per epoch)")print(f"best epoch       {result.best_epoch} of {config.epochs}")print(f"best val robust  {result.best_val_robust_acc:.4f}")print(f"checkpoint       {result.checkpoint_path}")

## 7. Trade-off curves

In [ ]:
import matplotlib.pyplot as pltimport pandas as pdhistory = pd.read_csv(result.history_path)figure_path = OUT_DIR / "adv_training_curves.png"fig, (ax_acc, ax_gap) = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")ax_acc.plot(history["epoch"], history["train_adv_acc"], label="train (adversarial)", color="tab:orange")ax_acc.plot(history["epoch"], history["val_clean_acc"], label="validation (clean)", color="tab:blue")ax_acc.plot(history["epoch"], history["val_robust_acc"], label="validation (robust)", color="tab:green")for milestone in config.milestone_epochs():    ax_acc.axvline(milestone, color="grey", linestyle=":", linewidth=1)ax_acc.axvline(result.best_epoch, color="tab:red", linestyle="--", linewidth=1,               label=f"selected (epoch {result.best_epoch})")ax_acc.set_xlabel("epoch")ax_acc.set_ylabel("accuracy")ax_acc.set_title("Accuracy and robustness during adversarial training")ax_acc.legend(loc="lower right", fontsize=9)ax_acc.grid(alpha=0.3)ax_gap.plot(history["epoch"], history["train_adv_acc"] - history["val_robust_acc"], color="tab:purple")ax_gap.axvline(result.best_epoch, color="tab:red", linestyle="--", linewidth=1)ax_gap.set_xlabel("epoch")ax_gap.set_ylabel("train adversarial minus validation robust")ax_gap.set_title("Robust generalisation gap")ax_gap.grid(alpha=0.3)fig.savefig(figure_path, dpi=150)plt.show()print(history.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 8. Evaluation of the selected checkpointRun once, on the test set, using the scoring functions from `experiments/robustness_eval.py` rather than areimplementation, so the table is directly comparable to the naturally trained baseline: same 1,000-image subset, sameeps and alpha, same definition of success (correct on clean input, wrong after the attack) and the same norm averagingover successes only.Those functions are underscore-prefixed, so importing them crosses a privacy boundary. Duplicating the protocol would beworse, because the two tables would stop being comparable the moment either definition moved. The follow-up is topromote them to public names in the module.Clean accuracy is additionally measured on the full 10,000-image test set, so it is comparable to the baseline's 93.43%rather than to a subset figure.Two expectations. Accuracy under PGD should be well above the baseline's 0.0000, and that gap is the result. C&Waccuracy need not reach zero: at a fixed penalty of c=1 it minimises distortion rather than forcing every flip, so readits success rate rather than its accuracy.

In [ ]:
import mathos.environ["CIFAR10_RESNET18_CKPT"] = str(result.checkpoint_path)from experiments.robustness_eval import (    ALPHA, CW_C, CW_STEPS, EPS, MODEL_NAME, PGD_RESTARTS, PGD_STEP_COUNTS,    Evaluation, _clean_predictions, _evaluate, _evaluations, _write_csv,)defended = load_model(DEVICE)loader = build_loader()clean_preds = _clean_predictions(defended, loader, DEVICE)table_path = OUT_DIR / "robustness_table_pgd_at.csv"rows: list[list[str]] = []for spec in _evaluations(defended):    res = _evaluate(defended, loader, DEVICE, spec, clean_preds)    rows.append([        MODEL_NAME, "pgd_at", spec.attack,        "" if math.isnan(spec.eps) else f"{spec.eps:.6f}",        str(spec.steps),        f"{res.accuracy:.4f}", f"{res.success_rate:.4f}",        f"{res.mean_linf:.6f}", f"{res.mean_l2:.6f}",    ])    tag = f"pgd-{spec.steps}" if spec.attack == "pgd" else spec.attack    print(f"{tag:>8}: accuracy = {res.accuracy:.4f}, success = {res.success_rate:.4f}, "          f"mean L-inf = {res.mean_linf:.6f}, mean L2 = {res.mean_l2:.6f}")_write_csv(rows, table_path)full_loader = build_loader(num_samples=10_000, batch_size=500)correct = total = 0with torch.no_grad():    for images, labels in full_loader:        images, labels = images.to(DEVICE), labels.to(DEVICE)        correct += int((defended(images).argmax(dim=1) == labels).sum().item())        total += labels.size(0)full_clean_acc = correct / totalprint(f"\nfull test clean accuracy: {full_clean_acc:.4f} over {total} images "      f"(naturally trained baseline: 0.9343)")print(f"written to {table_path}")print(f"protocol: eps={EPS:.6f}, alpha={ALPHA:.6f}, pgd steps={PGD_STEP_COUNTS}, restarts={PGD_RESTARTS}, "      f"cw c={CW_C} steps={CW_STEPS}, 1000 test images")

## 9. Obfuscated-gradient sanity checksA robustness number without these behind it is not one to report. Athalye, Carlini and Wagner identified fivecharacteristic behaviours of defences that merely make gradients uninformative; three are checkable here and are runbelow.**Unbounded attack.** With eps set to 1.0 the entire pixel box is reachable, so any model must fall to zero accuracy. Amodel that survives has a broken gradient rather than a defence, and this is the single most diagnostic check.**Monotonicity in eps.** Accuracy must fall as the budget grows. A non-monotone curve means the attack is not findingwhat a larger budget makes available.**Iterative beats single-step.** PGD-50 must be at least as damaging as PGD-20 and both at least as damaging as FGSM,which the table above already reports.The two remaining checks are not run here. A black-box or transfer attack must not outperform the white-box one, whichneeds the naturally trained checkpoint attached to craft transfer examples from, and random sampling must not beatgradient ascent. Both are cheap follow-ups and neither is satisfied by anything in this notebook.Failures are reported rather than raised, since the artefacts above are already written and a failure is a finding toinvestigate rather than a reason to discard the run.

In [ ]:
SWEEP_EPS = (2 / 255, 4 / 255, 8 / 255, 16 / 255)sweep: list[tuple[float, float]] = []for eps_value in SWEEP_EPS:    spec = Evaluation(        "pgd", 20, eps_value,        # eps_value bound as a default argument: Python closures capture by reference, so a        # bare reference would read the final loop value for every spec.        lambda x, y, e=eps_value: pgd(defended, x, y, e, e / 4, steps=20),    )    accuracy = _evaluate(defended, loader, DEVICE, spec, clean_preds).accuracy    sweep.append((eps_value, accuracy))    print(f"pgd-20 at {round(eps_value * 255):>2}/255: accuracy = {accuracy:.4f}")unbounded_spec = Evaluation(    "pgd", 50, 1.0, lambda x, y: pgd(defended, x, y, 1.0, 0.1, steps=50),)unbounded_acc = _evaluate(defended, loader, DEVICE, unbounded_spec, clean_preds).accuracyprint(f"pgd-50 unbounded (eps=1.0): accuracy = {unbounded_acc:.4f}")accuracies = [accuracy for _, accuracy in sweep]pgd_acc = {int(row[4]): float(row[5]) for row in rows if row[2] == "pgd"}fgsm_acc = next(float(row[5]) for row in rows if row[2] == "fgsm")checks = {    "unbounded attack reaches ~0 accuracy": unbounded_acc < 0.05,    "accuracy is monotone non-increasing in eps": all(a >= b for a, b in zip(accuracies, accuracies[1:])),    "pgd-50 at least as strong as pgd-20": pgd_acc[50] <= pgd_acc[20] + 1e-9,    "pgd-20 at least as strong as fgsm": pgd_acc[20] <= fgsm_acc + 1e-9,}print()for description, passed in checks.items():    print(f"  [{'PASS' if passed else 'FAIL'}]  {description}")if not all(checks.values()):    print("\nAt least one check failed. Do not report a robustness number from this run until it is explained.")

## 10. Output verification

In [ ]:
expected = [    result.history_path,    result.checkpoint_path,    OUT_DIR / "adv_training_config.json",    figure_path,    table_path,]missing = [path for path in expected if not path.is_file()]if missing:    raise RuntimeError(f"Missing expected outputs: {missing}")for path in expected:    print(f"{path.stat().st_size / 1024:>10.1f} KB  {path}")if SMOKE:    print("\nSMOKE run: these numbers are meaningless. Set SMOKE = False and run again.")else:    baseline = {"none": 0.9340, "fgsm": 0.1660, "pgd-20": 0.0000, "pgd-50": 0.0000, "cw_l2": 0.0000}    defended_acc = {(f"pgd-{row[4]}" if row[2] == "pgd" else row[2]): float(row[5]) for row in rows}    print("\naccuracy on the 1,000-image test subset, naturally trained -> adversarially trained")    for key, before in baseline.items():        after = defended_acc.get(key)        if after is not None:            print(f"  {key:>7}: {before:.4f} -> {after:.4f}  ({after - before:+.4f})")